# အသိအမှတ်ပြုခြင်းဒီဇိုင်းပုံစံ - သင်ခန်းစာ ၀၉


## တပ်ဆင်ခြင်း

ဤမှတ်စုစာမျက်နှာတွင် Microsoft Agent Framework ကို အသုံးပြုပြီး Metacognition ဒီဇိုင်းပုံစံကို ဖော်ပြထားသည်။

**လိုအပ်ချက်များ -**
- ပတ်ဝန်းကျင်မျိုးဆက်များမှတစ်ဆင့် ပြင်ဆင်ထားသည့် Azure OpenAI ဂျက်ရှင်းကို တပ်ဆင်ထားပြီးဖြစ်ရန်
- Azure CLI ဖြင့်အတည်ပြုထားခြင်း (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Metacognition ဆိုတာဘာလဲ?

Metacognition ဆိုတာက **စဉ်းစားခြင်းအပေါ်တွင် စဉ်းစားခြင်း** ဖြစ်ပါတယ်။ AI ကိုယ်စားလှယ်တွေအတွက် အဓိပ္ပာယ်မှာတော့:

- သူတို့ရဲ့ထွက်ရှိချက်တွေနဲ့ အတွေးအခေါ်စနစ်ကို **ကိုယ်တိုင်ပြန်လည်သုံးသပ်ခြင်း**
- **အမှားများကိုတွေ့ရှိ**ပြီး တိတ်ဆိတ်ပြီး မလုပ်ဆောင်ပျက်ကွက်ခြင်းမဟုတ်ဘဲ သက်သာစွာပြန်လည်ဖြေရှင်းနိုင်ခြင်း
- သူတို့ရဲ့ပြန်ကြားချက်တွေ တိကျပြီး အထောက်အကူဖြစ်မှုရှိမရှိကို **သုံးသပ်ခြင်း**
- စတင်အသုံးပြုမည့်နည်းလမ်း မအောင်မြင်ပါက သူတို့ရဲ့ မဟာဗျူဟာကို **ပြောင်းလဲခြင်း** (ဥပမာ- ဒုတိယစနစ်ကို အသုံးပြုခြင်း)

Metacognitive ကိုယ်စားလှယ်ဟာ မေးခွန်းတွေကိုသာဖြေမယ်ဆိုတာမဟုတ်ပါဘူး — ကိုယ်တိုင်၏ ဖော်ဆောင်မှုကို စောင့်ကြည့်ပြီး လိုအပ်သလို ပြင်ဆင်တတ်ပါတယ်။


## အဓိကနှင့် မူကားပစ္စည်းများ

ပုံမှန်မေတ္တအတွေး စံနမူနာတစ်ခုမှာ **နောက်ပြန်ရွေးခွင့် မူဝါဒ** ဖြစ်သည်။ အေးဂျင့်သည် ပထမဦးဆုံး အဓိကကိရိယာကို စမ်းသပ်ကြည့်သည်။ သင့်လျော်မှု မရရှိပါက (ဥပမာအားဖြင့် 404 အမှား)၊ အေးဂျင့်သည် တောင့်တင်းမှုကို သိရှိပြီး ပွင့်လင်းစွာ မူကားပစ္စည်းသို့ ပြောင်းရွှေ့သည်။

၎င်းသည် အဓိကဝန်ဆောင်မှုများ မရရှိနိုင်သည့် လက်တွေ့စနစ်များနှင့် ရောနှောထားခြင်းဖြစ်ပြီး အေးဂျင့်များသည် ပြဿနာကို ကိုယ်တိုင် ရှာဖွေသုံးသပ်ပြီး နောက်ထပ်လမ်းကြောင်းကို မရွေးချယ်မီ လုပ်ဆောင်ရပါသည်။

အောက်မှာ အပြေးခရီးရှာဖွေကိရိယာ နှစ်ခုကို သတ်မှတ်ထားသည်။
- **အဓိက** — ပါရီ၊ တိုကျို နှင့် ဘားဆီလိုနာကို ပါဝင်သည်
- **မူကားပစ္စည်း** — ဘာလင်၊ စစ်ဒနီး နှင့် နယူးယောက်မြို့ကို ပါဝင်သည်


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## အမှားပြန်လည်ကောင်းမွန်ရေးနှင့် ကိုယ်စိစစ်နိုင်သော ကိုယ်စားလှယ်

အောက်ပါ ကိုယ်စားလှယ်အား အဓိက ပျံသန်းစနစ်ကို ပထမဆုံး ကြိုးစားရန်၊ မအောင်မြင်မှုများကို သတိပြုရန် နှင့် ဖြတ်သန်းပြီး နောက်ခံစနစ်သို့ဖော်ပြန်ရန် ညွှန်ကြားထားသည်။ တစ် ချက်ချင်း ဖော်ပြချက်တိုင်းနောက်တွင် ၎င်းသည် အသုံးပြုသူ၏ မေးခွန်းကို လုံးဝ ဖြေကြားနိုင်ခြင်း ရှိ/မရှိကို အတိုချုံး ကိုယ်တိုင်သုံးသပ်သည်။


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## ကိုယ်ပိုင်သုံးသပ်မှုပုံစံ

metacognition ရဲ့ နောက်ထပ် အစိတ်အပိုင်းတစ်ခုကတော့ **ကိုယ်တိုင်သုံးသပ်ခြင်း** ဖြစ်သည်။ အဲဒါကတော့ မတူညီသော ကိုယ်တိုင်ဖြစ်စေ (သို့) တစ်ချက်ပြန်သွားပြီး ပြန်လည်သုံးသပ်သူအေဂျင့်တစ်ဦးက တုံ့ပြန်ချက်ကို ပြည့်စုံမှု၊ တိကျမှုနှင့် ရည်ရွယ်ချက်မှုအပေါ်မှာ ပြန်လည်သုံးသပ်သည်။

အောက်တွင် သယ်ယူပို့ဆောင်ရေးအေဂျင့်၏ တုံ့ပြန်ချက်များကို သုံးချက်ပေါ်မှာ အမှတ်ပေးသည့် `ResponseEvaluator` အေဂျင့်ကို ဖန်တီးပါမည်။


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## အနှစ်ချုပ်

ဒီသင်ခန်းစာမှာ Microsoft Agent Framework ကိုအသုံးပြုပြီး **မက်တာကိုဂျီနစ် အေးဂျင့်များ** ကို ဘယ်လိုဖန်တီးရမယ်ဆိုတာ သင်ယူခဲ့ပါတယ်။

- **ကိုယ်တိုင်ပြန်လည်စဉ်းစားခြင်း**: သူတို့ရဲ့ကိုယ်ပိုင် သဘောတရားများကို စောင့်ကြည့်ပြီး ဖြစ်ပျက်ပုံကို ပျောက်မပြောကြားတဲ့ အေးဂျင့်များ။
- **အမှားပြန်လည်ပြုပြင်ခြင်းနှင့် အကူအညီအကွဲများ**: အဓိက + ဘက်အပေါ်က စက်ပစ္စည်း ပုံစံဖြစ်ပြီး အေးဂျင့်သည် မလုပ်နိုင်ခြင်း အဖော်ပြချက်များ (ဥပမာ ၄၀၄ အမှား) ကို တွေ့ရှိပြီး အလိုအလျောက် အခြား အရင်းအမြစ်ကို စမ်းသပ်ကြည့်သည်။
- **ကိုယ်တိုင်အကဲဖြတ်ခြင်း**: ဖြေကြားချက်များကို အပြည့်အဝ၊ တိကျမှုနှင့် အကူအညီဖြစ်စေမှုများအတွက် အကဲဖြတ်သူအေးဂျင့် တစ်ဦး သီးသန့် ရှိသည်။

ဤပုံစံများသည် အေးဂျင့်များကို ပိုခိုင်မာ၊ ပိုရှင်းလင်းပြီး ယုံကြည်စိတ်ချရစေသည်။ ထိုကဲ့သို့သောအရည်အချင်းများသည် ထုတ်လုပ်မှုအသုံးပြုမှုများအတွက် အရေးကြီးသည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
